## Generator Function - Test

In [44]:
# imports 
import random
import uuid
import json
import pandas as pd
import numpy as np
import yaml
import warnings
warnings.filterwarnings("ignore")


In [45]:
# pull the config file
def read_config(config_path: str) -> dict:
    with open(config_path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)
    
config = read_config("../Config/config_sample.yml")
print(config)

{'outputLocation': './output', 'directory': './data', 'nSim': 100, 'tSpan': 365, 'populationSize': 1000, 'initialInfected': 10, 'seed': 12345, 'rho': 1.0, 'rcd': 14, 'ved': 7, 'rimmd': 180, 'generatorDistribution': 'configured', 'syntheticPopulation': {'static': {'age': {'categories': ['0-17', '18-49', '50-64', '65+'], 'weights': [0.22, 0.45, 0.2, 0.13]}, 'comorbidity': {'probabilityTrue': 0.25}, 'socialActivity': {'categories': ['high', 'medium', 'low'], 'weights': [0.25, 0.5, 0.25]}, 'geography': {'categories': ['urban', 'rural'], 'weights': [0.8, 0.2]}, 'mobility': {'categories': ['independent', 'assisted', 'immobile'], 'weights': [0.85, 0.12, 0.03]}, 'vaccineAcceptance': {'probabilityTrue': 0.75}}}, 'infectionRiskMultipliers': {'age': {'0-17': 0.75, '18-49': 1.0, '50-64': 1.0, '65+': 1.25}, 'comorbidity': {'true': 1.25, 'false': 1.0}, 'socialActivity': {'high': 1.25, 'medium': 1.0, 'low': 0.75}, 'geography': {'urban': 1.25, 'rural': 1.0}, 'mobility': {'independent': 1.0, 'assisted'

In [46]:
# grab template and keys
def load_json_schema(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def json_type_to_pandas_dtype(json_type):
    """
    Convert a JSON Schema type to a Pandas dtype.
    """

    if isinstance(json_type, list):
        json_type = [t for t in json_type if t != "null"]
        json_type = json_type[0] if json_type else "object"

    type_mapping = {
        "string": "string",
        "integer": "Int64",
        "number": "float64",
        "boolean": "boolean",
        "array": "object",
        "object": "object",
    }

    return type_mapping.get(json_type, "object")


def flatten_json_schema(schema: dict, parent_key: str = "") -> list[dict]:
    """
    Recursively flatten a nested JSON Schema into table-style metadata.
    """

    rows = []

    properties = schema.get("properties", {})
    required_fields = set(schema.get("required", []))

    for field_name, field_schema in properties.items():
        full_name = f"{parent_key}.{field_name}" if parent_key else field_name

        field_type = field_schema.get("type")

        # If this is a nested object, recurse into its properties
        if field_type == "object" and "properties" in field_schema:
            rows.extend(flatten_json_schema(field_schema, full_name))

        else:
            rows.append(
                {
                    "column": full_name,
                    "json_type": field_type,
                    "pandas_dtype": json_type_to_pandas_dtype(field_type),
                    "required": field_name in required_fields,
                    "description": field_schema.get("description"),
                    "enum": field_schema.get("enum"),
                    "minimum": field_schema.get("minimum"),
                    "format": field_schema.get("format"),
                }
            )

    return rows


schema = load_json_schema("person_template.json")

schema_columns = flatten_json_schema(schema)

schema_df

,column,json_type,pandas_dtype,required,description,enum,minimum,format
0,static.guid,string,string,True,Unique identifier for the individual.,None,NaN,uuid
1,static.age,string,string,True,Age range of the individual.,"[0-17, 18-49, 50-64, 65+]",NaN,None
2,static.comorbidity,boolean,boolean,True,Indicates whether the individual has one or mo...,None,NaN,None
3,static.socialActivity,string,string,True,General level of social activity.,"[high, medium, low]",NaN,None
4,static.geography,string,string,True,Geographic setting of the individual.,"[urban, rural]",NaN,None
5,static.mobility,string,string,True,Mobility status of the individual.,"[independent, assisted, immobile]",NaN,None
6,static.vaccineAcceptance,boolean,boolean,True,Indicates whether the individual is willing to...,None,NaN,None
7,dynamic.vaccineStatus,boolean,boolean,True,Indicates whether the individual is currently ...,None,NaN,None
8,dynamic.proactiveVaccine,boolean,boolean,True,Indicates whether the individual proactively r...,None,NaN,None
9,dynamic.numberOfInfections,integer,Int64,True,Total number of infections experienced by the ...,None,0.0,None


In [80]:
#grab stuff from the yml
populationSize = config["populationSize"]
age_config = config["syntheticPopulation"]["static"]["age"]
comorbidity_config = config["syntheticPopulation"]["static"]["comorbidity"]
social_activity_config = config["syntheticPopulation"]["static"]["socialActivity"]
geography_config = config["syntheticPopulation"]["static"]["geography"]
mobility_config = config["syntheticPopulation"]["static"]["mobility"]
vaccine_acceptance_config = config["syntheticPopulation"]["static"]["vaccineAcceptance"]

infection_risk_multipliers = config["infectionRiskMultipliers"]

seed = config["seed"]
rng =np.random.default_rng(config["seed"])

In [81]:
# begin random generation of data based on the schema

def generate_guid() -> str:
    return str(uuid.uuid4())


def generate_age_str() -> str:
    return rng.choice(
        age_config["categories"],
        p=age_config["weights"]
    )


def generate_comorbidity() -> bool:
    return bool(
        rng.choice(
            [True, False],
            p=[
                comorbidity_config["probabilityTrue"],
                1 - comorbidity_config["probabilityTrue"]
            ]
        )
    )


def generate_social_activity() -> str:
    return rng.choice(
        social_activity_config["categories"],
        p=social_activity_config["weights"]
    )


def generate_geography() -> str:
    return rng.choice(
        geography_config["categories"],
        p=geography_config["weights"]
    )


def generate_mobility() -> str:
    return rng.choice(
        mobility_config["categories"],
        p=mobility_config["weights"]
    )


def generate_vaccine_acceptance() -> bool:
    return bool(
        rng.choice(
            [True, False],
            p=[
                vaccine_acceptance_config["probabilityTrue"],
                1 - vaccine_acceptance_config["probabilityTrue"]
            ]
        )
    )


def get_risk_multiplier(field_name: str, field_value) -> float:
    if isinstance(field_value, bool):
        lookup_value = str(field_value).lower()
    else:
        lookup_value = field_value

    return infection_risk_multipliers[field_name][lookup_value]


# main
rows = []

for _ in range(populationSize):
    age = generate_age_str()
    comorbidity = generate_comorbidity()
    social_activity = generate_social_activity()
    geography = generate_geography()
    mobility = generate_mobility()
    vaccine_acceptance = generate_vaccine_acceptance()

    row = {
        "static.guid": generate_guid(),

        "static.age": age,
        "static.ageRiskMultiplier": get_risk_multiplier("age", age),

        "static.comorbidity": comorbidity,
        "static.comorbidityRiskMultiplier": get_risk_multiplier("comorbidity", comorbidity),

        "static.socialActivity": social_activity,
        "static.socialActivityRiskMultiplier": get_risk_multiplier("socialActivity", social_activity),

        "static.geography": geography,
        "static.geographyRiskMultiplier": get_risk_multiplier("geography", geography),

        "static.mobility": mobility,
        "static.mobilityRiskMultiplier": get_risk_multiplier("mobility", mobility),

        "static.vaccineAcceptance": vaccine_acceptance,
        "static.vaccineAcceptanceRiskMultiplier": get_risk_multiplier("vaccineAcceptance", vaccine_acceptance),

        "dynamic.vaccineStatus": False,
        "dynamic.proactiveVaccine": False,
        "dynamic.numberOfInfections": 0,
        "dynamic.sirvStatus": "S",
        "dynamic.infectedDays": 0,
        "dynamic.vaccinatedDays": 0,
        "dynamic.recoveredDays": 0,
        "dynamic.currentLocation.xcor": 0.0,
        "dynamic.currentLocation.ycor": 0.0
    }

    rows.append(row)

data_df = pd.DataFrame(rows)

data_df

,static.guid,static.age,static.ageRiskMultiplier,static.comorbidity,static.comorbidityRiskMultiplier,static.socialActivity,static.socialActivityRiskMultiplier,static.geography,static.geographyRiskMultiplier,static.mobility,...,static.vaccineAcceptanceRiskMultiplier,dynamic.vaccineStatus,dynamic.proactiveVaccine,dynamic.numberOfInfections,dynamic.sirvStatus,dynamic.infectedDays,dynamic.vaccinatedDays,dynamic.recoveredDays,dynamic.currentLocation.xcor,dynamic.currentLocation.ycor
0,dc2b5966-4313-4a89-b027-543266bd96f6,18-49,1.00,False,1.00,low,0.75,urban,1.25,independent,...,0.75,False,False,0,S,0,0,0,0.0,0.0
1,09a8f04d-6aae-487d-96c1-3d5392210b29,18-49,1.00,True,1.25,medium,1.00,rural,1.00,independent,...,1.00,False,False,0,S,0,0,0,0.0,0.0
2,6ec46a94-b925-4bbb-8d7f-0bafb048424e,18-49,1.00,True,1.25,medium,1.00,rural,1.00,independent,...,0.75,False,False,0,S,0,0,0,0.0,0.0
3,ef4ff62b-100d-44ca-ad3a-5591fcfa623b,50-64,1.00,True,1.25,high,1.25,urban,1.25,independent,...,0.75,False,False,0,S,0,0,0,0.0,0.0
4,525a278e-52f3-49f9-b66f-3f7ee25c5cdb,18-49,1.00,False,1.00,high,1.25,urban,1.25,independent,...,0.75,False,False,0,S,0,0,0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,d2d3b368-b746-455e-b1c5-82518a0b3c2a,18-49,1.00,False,1.00,medium,1.00,urban,1.25,independent,...,0.75,False,False,0,S,0,0,0,0.0,0.0
996,144ecd4f-f3c5-4b3c-8f8a-e7015bca4aaa,50-64,1.00,False,1.00,medium,1.00,urban,1.25,independent,...,0.75,False,False,0,S,0,0,0,0.0,0.0
997,f4cbf9ed-a076-4665-8437-6cf31d43cb47,50-64,1.00,False,1.00,medium,1.00,urban,1.25,immobile,...,0.75,False,False,0,S,0,0,0,0.0,0.0
998,4cbbff5d-00ad-4af6-9793-f50953d22831,0-17,0.75,False,1.00,high,1.25,urban,1.25,independent,...,0.75,False,False,0,S,0,0,0,0.0,0.0


In [82]:
data_df.to_csv("synthetic_population.csv", index=False)

## Infection and Positioning

In [86]:
# infect some people and see how it goes

numb_to_infect = config["initialInfected"]

infection_indices = rng.choice(data_df.index, size=numb_to_infect, replace=False).tolist()

for idx in infection_indices:
    data_df.at[idx, "dynamic.sirvStatus"] = "I"

In [87]:
#sanity check
print(infection_indices)
print(data_df.loc[infection_indices, ["static.guid", "dynamic.sirvStatus"]])

[202, 985, 329, 257, 965, 488, 505, 376, 731, 756]
                              static.guid dynamic.sirvStatus
202  03917ca1-8734-41ef-a755-b9418d25a507                  I
985  ab87532a-b66e-4bb8-a39a-2ba2b5957058                  I
329  09887d95-b28a-467c-92d5-4e8f6461fcfd                  I
257  3745124a-879a-485b-b642-40a64550ccca                  I
965  05ac467b-20c8-4aab-a99b-5a9f4bb0835f                  I
488  8ceb76b1-b44c-4ae7-afe1-94c871bb3e1a                  I
505  9c6ae21f-7cb7-4747-b8e7-f1bc4dea7f8d                  I
376  7acafda5-7348-4de5-9b4c-7c40b78a748c                  I
731  c72531f3-5acd-4fac-893f-558a6bcb0cfb                  I
756  07b50385-96e7-4de7-a01e-1d94e56d74c9                  I
